# Sesión 05 — Selección de Características y Reducción de Dimensionalidad
### Reconocimiento de Patrones y Aprendizaje Automático — Posgrado en Ingeniería Biomédica

**Módulo I · Fundamentos del Aprendizaje Estadístico**

## Objetivos de aprendizaje

Al finalizar esta sesión serás capaz de:

1. Aplicar PCA, construir la gráfica de codos y cuantificar el error de reconstrucción.
2. Comparar PCA (no supervisado) con LDA (supervisado) para reducción de dimensionalidad.
3. Aplicar ICA para separación ciega de fuentes y eliminación de artefactos EEG.
4. Implementar y comparar métodos de selección de características: filtro por información mutua, ANOVA y eliminación recursiva de características (RFE).
5. Demostrar cuantitativamente la fuga de información en la selección de características dentro del pipeline de validación cruzada.

## Lecturas recomendadas

| Prioridad | Referencia |
|---|---|
| ★★★ | Bishop (2006). *PRML*. §12.1 (PCA), §12.2 (Factor Analysis). Springer. |
| ★★★ | Hyvärinen, A., Karhunen, J. & Oja, E. (2001). *Independent Component Analysis*. Wiley. Cap. 7–8. |
| ★★☆ | Guyon, I. & Elisseeff, A. (2003). An introduction to variable and feature selection. *JMLR*, 3, 1157–1182. |
| ★★☆ | Delorme, A. & Makeig, S. (2004). EEGLAB: an open source toolbox for EEG analysis. *Journal of Neuroscience Methods*, 134(1), 9–21. https://doi.org/10.1016/j.jneumeth.2003.10.009 |
| ★☆☆ | Wold, S., Esbensen, K. & Geladi, P. (1987). Principal component analysis. *Chemometrics and Intelligent Laboratory Systems*, 2(1–3), 37–52. |

## Parte 0 — Configuración

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy import stats, linalg

rng = np.random.default_rng(42)

plt.rcParams.update({
    'figure.dpi': 120,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': True,
    'grid.alpha': 0.3,
    'font.size': 11,
})
print('Configuración completa.')

## Parte 1 — Análisis de Componentes Principales (PCA)

PCA encuentra las direcciones de máxima varianza en los datos.
Matemáticamente, busca la matriz ortogonal $\mathbf{W}$ tal que
$\mathbf{Z} = \mathbf{X}\mathbf{W}$ maximiza la varianza de las columnas de $\mathbf{Z}$.

La solución es la descomposición en valores propios de la matriz de covarianza:

$$\mathbf{\Sigma} = \frac{1}{N-1}\mathbf{X}^\top\mathbf{X} = \mathbf{W}\mathbf{\Lambda}\mathbf{W}^\top$$

Las columnas de $\mathbf{W}$ son los **vectores propios** (componentes principales),
y los elementos de la diagonal de $\mathbf{\Lambda}$ son los **valores propios**
(varianza explicada por cada componente).

In [ ]:
# ── PCA desde cero aplicado a señales EEG de 64 canales ──────────────────────
# Simulamos un dataset EEG multivariado con estructura de covarianza realista
# Diseño inspirado en: Blankertz, B. et al. (2007). IEEE TBME 54(12):2141–2150.

N_muestras = 500
N_canales  = 64

# Crear covarianza realista: bloques correlacionados (regiones cerebrales)
def covarianza_eeg(n_canales, n_bloques=8, rng_=None):
    rng_ = rng_ or np.random.default_rng()
    bloque = n_canales // n_bloques
    Sigma  = np.eye(n_canales) * 0.3
    for b in range(n_bloques):
        i0, i1 = b*bloque, (b+1)*bloque
        rho = rng_.uniform(0.4, 0.8)
        Sigma[i0:i1, i0:i1] = rho + (1-rho)*np.eye(bloque)
    # Asegurar que sea definida positiva
    Sigma = Sigma @ Sigma.T
    return Sigma / np.diag(Sigma).max()

Sigma_eeg = covarianza_eeg(N_canales, rng_=rng)
L = np.linalg.cholesky(Sigma_eeg + 1e-6*np.eye(N_canales))
X_eeg = rng.normal(0, 1, (N_muestras, N_canales)) @ L.T

# ── PCA manual ────────────────────────────────────────────────────────────────
def pca_manual(X):
    """PCA mediante descomposición en valores propios de la covarianza."""
    X_c   = X - X.mean(axis=0)                           # centrar
    Sigma = X_c.T @ X_c / (len(X) - 1)                  # covarianza
    vals, vecs = np.linalg.eigh(Sigma)                   # valores y vectores propios
    orden = np.argsort(vals)[::-1]                        # ordenar descendente
    vals  = vals[orden]
    vecs  = vecs[:, orden]
    var_exp = vals / vals.sum()                           # varianza explicada
    Z       = X_c @ vecs                                  # proyección
    return Z, vecs, vals, var_exp

Z, W, eigenvals, var_exp = pca_manual(X_eeg)
var_cum = np.cumsum(var_exp)

# Número de componentes para 95% de varianza explicada
k_95 = np.searchsorted(var_cum, 0.95) + 1
k_99 = np.searchsorted(var_cum, 0.99) + 1
print(f'Componentes para 90% de varianza: {np.searchsorted(var_cum,0.90)+1}')
print(f'Componentes para 95% de varianza: {k_95}')
print(f'Componentes para 99% de varianza: {k_99}')
print(f'Reducción: {N_canales} → {k_95} canales ({100*(1-k_95/N_canales):.0f}% reducción)')

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# Gráfica de codos
axes[0].bar(range(1, 21), var_exp[:20] * 100, color='steelblue', alpha=0.8)
axes[0].plot(range(1, 21), var_exp[:20] * 100, 'ko-', ms=4)
axes[0].set(xlabel='Componente principal', ylabel='Varianza explicada (%)',
            title='Gráfica de codos (scree plot)\nPrimeras 20 de 64 componentes')

# Varianza acumulada
axes[1].plot(range(1, N_canales+1), var_cum * 100, 'steelblue', lw=2)
axes[1].axhline(95, color='tomato', ls='--', lw=1.5, label=f'95% → {k_95} CPs')
axes[1].axhline(99, color='seagreen', ls='--', lw=1.5, label=f'99% → {k_99} CPs')
axes[1].axvline(k_95, color='tomato', ls=':', lw=1)
axes[1].axvline(k_99, color='seagreen', ls=':', lw=1)
axes[1].set(xlabel='Número de componentes', ylabel='Varianza acumulada (%)',
            title='Varianza acumulada explicada')
axes[1].legend(fontsize=9)

# Error de reconstrucción vs número de componentes
X_c = X_eeg - X_eeg.mean(axis=0)
errores_rec = []
for k in range(1, N_canales+1):
    X_rec = (X_c @ W[:, :k]) @ W[:, :k].T
    errores_rec.append(np.mean((X_c - X_rec)**2))

axes[2].semilogy(range(1, N_canales+1), errores_rec, 'steelblue', lw=2)
axes[2].axvline(k_95, color='tomato', ls='--', lw=1.5, label=f'{k_95} CPs (95%)')
axes[2].set(xlabel='Número de componentes', ylabel='Error de reconstrucción (log)',
            title='Error de reconstrucción vs k')
axes[2].legend(fontsize=9)

plt.suptitle(f'PCA en señal EEG simulada de {N_canales} canales — {N_muestras} muestras',
              fontsize=12, y=1.01)
plt.tight_layout()
plt.show()

## Parte 2 — PCA vs LDA: reducción supervisada vs no supervisada

PCA maximiza la varianza total — no usa información de las etiquetas.
El **Análisis Discriminante Lineal (LDA)** maximiza la separabilidad entre clases:

$$\mathbf{w}^* = \arg\max_\mathbf{w} \frac{\mathbf{w}^\top \mathbf{S}_B \mathbf{w}}{\mathbf{w}^\top \mathbf{S}_W \mathbf{w}}$$

donde $\mathbf{S}_B$ es la dispersión entre clases y $\mathbf{S}_W$ es la dispersión intra-clase.
La solución es el problema generalizado de valores propios $\mathbf{S}_B\mathbf{w} = \lambda \mathbf{S}_W\mathbf{w}$.

**Regla práctica:** para clasificación, LDA casi siempre supera a PCA porque usa
la información de las etiquetas. PCA es útil para visualización, compresión y
preprocesamiento sin etiquetas.

In [ ]:
# ── PCA vs LDA — clasificación de estados mentales EEG ───────────────────────
# 3 clases: reposo, imaginería motora derecha, imaginería motora izquierda
# 30 características espectrales (bandas delta, theta, alpha, beta, gamma × 6 canales)

N_por_clase = 80
N_feats     = 30
K_clases    = 3

# Simular datos con separabilidad moderada entre clases
medias = rng.normal(0, 1.5, (K_clases, N_feats))
X_lda  = np.vstack([rng.normal(medias[k], 1.0, (N_por_clase, N_feats))
                     for k in range(K_clases)])
y_lda  = np.repeat(np.arange(K_clases), N_por_clase)
nombres_clases = ['Reposo', 'IM Derecha', 'IM Izquierda']

def lda_manual(X, y):
    """LDA para K clases, proyecta a K-1 dimensiones."""
    clases  = np.unique(y)
    K       = len(clases)
    n_total = len(y)
    mu_g    = X.mean(axis=0)

    # Dispersión entre clases S_B
    S_B = np.zeros((X.shape[1], X.shape[1]))
    for k in clases:
        n_k = (y == k).sum()
        mu_k = X[y == k].mean(axis=0)
        diff  = (mu_k - mu_g).reshape(-1, 1)
        S_B  += n_k * diff @ diff.T

    # Dispersión intra-clase S_W
    S_W = np.zeros_like(S_B)
    for k in clases:
        Xk   = X[y == k]
        mu_k = Xk.mean(axis=0)
        S_W += (Xk - mu_k).T @ (Xk - mu_k)

    # Problema generalizado de valores propios
    S_W_reg = S_W + 1e-6 * np.eye(S_W.shape[0])   # regularización
    vals, vecs = linalg.eigh(S_B, S_W_reg)
    orden = np.argsort(vals)[::-1]
    return vecs[:, orden[:K-1]], vals[orden[:K-1]]

# Aplicar PCA y LDA
Z_pca, W_pca, _, var_pca = pca_manual(X_lda)
W_lda, vals_lda = lda_manual(X_lda, y_lda)
Z_lda = (X_lda - X_lda.mean(axis=0)) @ W_lda

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
colores_k = ['steelblue', 'tomato', 'seagreen']

for ax, (Z, titulo) in zip(axes, [
    (Z_pca[:, :2], f'PCA — CP1 ({var_pca[0]*100:.1f}%) vs CP2 ({var_pca[1]*100:.1f}%)\nNo usa etiquetas — maximiza varianza total'),
    (Z_lda,        'LDA — Discriminante 1 vs Discriminante 2\nUsa etiquetas — maximiza separabilidad entre clases'),
]):
    for k, (nombre, color) in enumerate(zip(nombres_clases, colores_k)):
        mask = y_lda == k
        ax.scatter(Z[mask, 0], Z[mask, 1], alpha=0.5, s=20,
                    color=color, label=nombre)
    ax.set(xlabel='Dimensión 1', ylabel='Dimensión 2', title=titulo)
    ax.legend(fontsize=9)

plt.suptitle('PCA vs LDA — proyección de características EEG a 2D',
              fontsize=12, y=1.01)
plt.tight_layout()
plt.show()

# Evaluar exactitud con clasificador de centroide
def exactitud_centroide_cv(Z, y, k_cv=5):
    N   = len(y)
    idx = rng.permutation(N)
    sz  = N // k_cv
    accs = []
    for k in range(k_cv):
        te = idx[k*sz:(k+1)*sz]
        tr = np.concatenate([idx[:k*sz], idx[(k+1)*sz:]])
        centroides = np.array([Z[tr][y[tr]==c].mean(0) for c in range(K_clases)])
        dists = np.linalg.norm(Z[te, None] - centroides[None], axis=2)
        pred  = dists.argmin(axis=1)
        accs.append((pred == y[te]).mean())
    return np.mean(accs)

acc_pca = exactitud_centroide_cv(Z_pca[:, :2], y_lda)
acc_lda = exactitud_centroide_cv(Z_lda,         y_lda)
acc_all = exactitud_centroide_cv(X_lda,          y_lda)

print(f'Exactitud (k-fold, clasificador centroide):')
print(f'  PCA (2 componentes):  {acc_pca:.3f}')
print(f'  LDA (2 discrimin.):   {acc_lda:.3f}')
print(f'  Todas las feats (30): {acc_all:.3f}')

## Parte 3 — ICA: separación ciega de fuentes y artefactos EEG

ICA asume que las señales observadas $\mathbf{x}$ son mezclas lineales de fuentes
estadísticamente independientes $\mathbf{s}$:

$$\mathbf{x} = \mathbf{A}\mathbf{s} \qquad \Rightarrow \qquad \mathbf{s} = \mathbf{W}\mathbf{x}$$

donde $\mathbf{A}$ es la **matriz de mezcla** (desconocida) y $\mathbf{W} = \mathbf{A}^{-1}$
es la **matriz de separación**.

La independencia estadística se maximiza minimizando la información mutua entre
las fuentes, lo que equivale a maximizar la no-gaussianidad.

**Aplicación clave en EEG:** los artefactos de parpadeo ocular (EOG), cardiacos (ECG)
y musculares (EMG) son fuentes independientes de la actividad neuronal.
ICA permite separarlos y eliminarlos sin afectar la señal de interés.

> **Referencia:** Delorme, A. & Makeig, S. (2004). EEGLAB: an open source toolbox
> for analysis of single-trial EEG dynamics. *Journal of Neuroscience Methods*,
> 134(1), 9–21. https://doi.org/10.1016/j.jneumeth.2003.10.009

In [ ]:
# ── ICA para separación de artefactos EEG ────────────────────────────────────
# Simulamos 4 canales EEG mezclados con:
#   s1: actividad neuronal (oscilación alfa 10 Hz)
#   s2: artefacto de parpadeo EOG (pulsos lentos)
#   s3: artefacto cardiaco ECG (picos QRS)
#   s4: ruido de fondo

T      = 1000   # muestras
t_axis = np.linspace(0, 4, T)   # 4 segundos
fs     = 250    # Hz

# Fuentes independientes
s1_alfa = np.sin(2 * np.pi * 10 * t_axis) * (1 + 0.3*np.sin(2*np.pi*0.5*t_axis))
s2_eog  = np.zeros(T)
for t_blink in [0.5, 1.8, 3.2]:
    centro = int(t_blink * fs)
    s2_eog += np.exp(-((np.arange(T) - centro)**2) / (2*(0.08*fs)**2)) * 3.0
s3_ecg  = np.zeros(T)
for t_qrs in np.arange(0.1, 4.0, 0.75):
    centro = int(t_qrs * fs)
    s3_ecg += (np.exp(-((np.arange(T)-centro)**2)/(2*(0.02*fs)**2)) -
               0.3*np.exp(-((np.arange(T)-centro-8)**2)/(2*(0.04*fs)**2))) * 1.5
s4_ruido = rng.normal(0, 0.4, T)

S_fuentes = np.vstack([s1_alfa, s2_eog, s3_ecg, s4_ruido]).T   # (T, 4)

# Normalizar fuentes
S_fuentes = S_fuentes / S_fuentes.std(axis=0, keepdims=True)

# Matriz de mezcla aleatoria (simula la propagación de fuentes a electrodos)
A_mezcla = rng.normal(0, 1, (4, 4))
A_mezcla /= np.abs(A_mezcla).max()
X_mezcla  = S_fuentes @ A_mezcla.T   # (T, 4) — señales observadas en los electrodos

# ── FastICA simplificado (un paso de Newton) ──────────────────────────────────
def fastica_simple(X, n_comp=None, max_iter=200, tol=1e-5):
    """
    FastICA simplificado usando la función de contraste g(u) = tanh(u).
    Retorna: S_estimada (T, n_comp), W_separacion (n_comp, n_feats)
    """
    X = X.copy()
    n_samples, n_feats = X.shape
    n_comp = n_comp or n_feats

    # Centrar
    X -= X.mean(axis=0)

    # Blanquear: transformar para que la covarianza sea identidad
    U, D, Vt = np.linalg.svd(X, full_matrices=False)
    X_blanco  = U * np.sqrt(n_samples)   # (T, n_feats)

    W = rng.normal(0, 1, (n_comp, n_feats))   # inicialización aleatoria

    for comp in range(n_comp):
        w = W[comp].copy()
        w /= np.linalg.norm(w)

        for _ in range(max_iter):
            u    = X_blanco @ w                    # proyección
            g    = np.tanh(u)                      # g(u) = tanh(u)
            g_pr = 1 - g**2                        # g'(u)
            # Actualización FastICA
            w_new = (X_blanco.T @ g) / n_samples - g_pr.mean() * w
            # Decorrelación (Gram-Schmidt)
            for j in range(comp):
                w_new -= (w_new @ W[j]) * W[j]
            w_new /= np.linalg.norm(w_new)
            # Verificar convergencia
            if np.abs(np.abs(w_new @ w) - 1) < tol:
                break
            w = w_new
        W[comp] = w

    # Deshacer blanqueo para obtener separación en espacio original
    W_final  = W @ (Vt.T * (1/D * np.sqrt(n_samples))).T
    S_estim  = X_blanco @ W.T
    return S_estim, W_final

S_estimada, W_sep = fastica_simple(X_mezcla.copy(), n_comp=4)

# Identificar componente de parpadeo (mayor correlación con s2_eog)
corr_eog = [np.abs(np.corrcoef(S_estimada[:, i], s2_eog)[0,1])
             for i in range(4)]
idx_eog = np.argmax(corr_eog)
print(f'Componente ICA más correlacionada con EOG: IC{idx_eog+1}'
       f' (r={corr_eog[idx_eog]:.3f})')

# Reconstruir señal sin el componente de artefacto
S_limpia = S_estimada.copy()
S_limpia[:, idx_eog] = 0   # suprimir componente EOG

# Reconstruir en espacio de electrodos
A_estim  = np.linalg.pinv(W_sep)   # A ≈ W⁻¹
X_limpia = (S_limpia @ W_sep) + X_mezcla.mean(axis=0)

fig, axes = plt.subplots(4, 3, figsize=(15, 10), sharex=True)
nombres_fuentes = ['Alfa (neuronal)', 'EOG (parpadeo)', 'ECG (cardiaco)', 'Ruido']
nombres_canal   = [f'Canal {i+1}' for i in range(4)]
t_plot = t_axis[:400]

for i in range(4):
    # Columna 1: fuentes originales
    axes[i,0].plot(t_plot, S_fuentes[:400,i], lw=1.2,
                    color='steelblue' if i==0 else 'tomato' if i==1
                    else 'seagreen' if i==2 else 'gray')
    axes[i,0].set_ylabel(nombres_fuentes[i], fontsize=9)
    axes[i,0].set_yticks([])
    if i==0: axes[i,0].set_title('Fuentes reales\n(desconocidas)', fontsize=10)

    # Columna 2: señales mezcladas
    axes[i,1].plot(t_plot, X_mezcla[:400,i], lw=1, color='dimgray')
    axes[i,1].set_ylabel(nombres_canal[i], fontsize=9)
    axes[i,1].set_yticks([])
    if i==0: axes[i,1].set_title('Señales mezcladas\n(electrodos EEG)', fontsize=10)

    # Columna 3: componentes ICA estimadas
    color_ic = 'tomato' if i == idx_eog else 'seagreen'
    axes[i,2].plot(t_plot, S_estimada[:400,i], lw=1.2, color=color_ic)
    lbl = f'IC{i+1}' + (' ← EOG' if i==idx_eog else '')
    axes[i,2].set_ylabel(lbl, fontsize=9)
    axes[i,2].set_yticks([])
    if i==0: axes[i,2].set_title('Componentes ICA\n(fuentes estimadas)', fontsize=10)

for ax in axes[-1, :]:
    ax.set_xlabel('Tiempo (s)')

plt.suptitle('FastICA — Separación ciega de fuentes en EEG\n'
              'El componente EOG (parpadeo) se identifica y suprime sin afectar la señal alfa',
              fontsize=12, y=1.01)
plt.tight_layout()
plt.show()

## Parte 4 — Métodos de selección de características

Existen tres familias de métodos de selección de características:

| Familia | Método | Ventaja | Limitación |
|---|---|---|---|
| **Filtro** | Información mutua, ANOVA | Rápido, independiente del modelo | No considera interacciones |
| **Envolvente** | RFE, forward/backward selection | Considera interacciones | Costoso computacionalmente |
| **Incrustado** | LASSO, Random Forest importance | Simultáneo con el entrenamiento | Específico del modelo |

In [ ]:
# ── Comparación de métodos de selección de características ────────────────────
# Dataset: características de HRV (variabilidad de frecuencia cardiaca)
# para clasificación estrés vs reposo
# Inspirado en: Schmidt, P. et al. (2018). Introducing WESAD, a multimodal dataset
# for wearable stress and affect detection. ACM ICMI 2018.
# https://doi.org/10.1145/3242969.3242985

N_samp_hrv = 200
N_feats_hrv = 40   # características de dominio tiempo, frecuencia y no lineal
N_relevantes = 10  # características genuinamente informativas

# Datos simulados: N_relevantes características discriminativas + ruido
y_hrv = rng.integers(0, 2, N_samp_hrv)
X_hrv = rng.normal(0, 1, (N_samp_hrv, N_feats_hrv))
# Características relevantes: diferencia de media según la clase
idx_relevantes = rng.choice(N_feats_hrv, N_relevantes, replace=False)
for idx in idx_relevantes:
    X_hrv[:, idx] += y_hrv * rng.uniform(0.8, 1.5)

nombres_feats = [f'HRV_{i:02d}' for i in range(N_feats_hrv)]

# ── Método 1: Filtro por información mutua ────────────────────────────────────
def informacion_mutua_discreta(x, y, n_bins=10):
    """Aproximación de la información mutua por discretización."""
    x_disc = np.digitize(x, np.linspace(x.min(), x.max(), n_bins))
    prob_xy = np.zeros((n_bins+1, 2))
    for xi, yi in zip(x_disc, y.astype(int)):
        prob_xy[xi, yi] += 1
    prob_xy /= len(y)
    prob_x = prob_xy.sum(axis=1, keepdims=True)
    prob_y = prob_xy.sum(axis=0, keepdims=True)
    with np.errstate(divide='ignore', invalid='ignore'):
        ratio = np.where(prob_xy > 0, prob_xy / (prob_x * prob_y + 1e-10), 0)
        im = np.sum(prob_xy * np.where(ratio > 0, np.log(ratio + 1e-10), 0))
    return max(0, im)

scores_im = np.array([informacion_mutua_discreta(X_hrv[:,j], y_hrv)
                        for j in range(N_feats_hrv)])

# ── Método 2: ANOVA F-test ────────────────────────────────────────────────────
scores_anova = np.array([stats.f_oneway(X_hrv[y_hrv==0, j],
                                          X_hrv[y_hrv==1, j]).statistic
                           for j in range(N_feats_hrv)])

# ── Método 3: RFE con clasificador de centroide ───────────────────────────────
def rfe_centroide(X, y, n_top):
    """Eliminación recursiva: en cada paso elimina la característica menos útil."""
    feats_activas = list(range(X.shape[1]))
    ranking = []

    while len(feats_activas) > n_top:
        X_sub = X[:, feats_activas]
        # Importancia = diferencia de medias entre clases
        importancias = np.abs(X_sub[y==1].mean(0) - X_sub[y==0].mean(0))
        peor = np.argmin(importancias)
        ranking.insert(0, feats_activas[peor])
        feats_activas.pop(peor)

    ranking = feats_activas + ranking  # activas primero
    return np.array(ranking)

ranking_rfe = rfe_centroide(X_hrv, y_hrv, n_top=N_relevantes)
scores_rfe  = np.zeros(N_feats_hrv)
for rank, feat in enumerate(ranking_rfe):
    scores_rfe[feat] = N_feats_hrv - rank

# ── Evaluación: ¿qué método recupera mejor las características relevantes? ────
def precision_at_k(scores, idx_relevantes, k):
    top_k = np.argsort(scores)[-k:]
    return len(set(top_k) & set(idx_relevantes)) / k

print('Precisión en la recuperación de características relevantes:')
print(f'{"Método":<25}  {"Top-5":>6}  {"Top-10":>7}  {"Top-15":>7}')
print('─' * 48)
for nombre, scores in [('Info. Mutua', scores_im),
                         ('ANOVA',       scores_anova),
                         ('RFE',         scores_rfe)]:
    p5  = precision_at_k(scores, idx_relevantes, 5)
    p10 = precision_at_k(scores, idx_relevantes, 10)
    p15 = precision_at_k(scores, idx_relevantes, 15)
    print(f'{nombre:<25}  {p5:>6.1%}  {p10:>7.1%}  {p15:>7.1%}')

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, (nombre, scores) in zip(axes,
    [('Info. Mutua', scores_im), ('ANOVA F', scores_anova), ('RFE', scores_rfe)]):
    colores_bar = ['tomato' if i in idx_relevantes else 'steelblue'
                    for i in range(N_feats_hrv)]
    ax.bar(range(N_feats_hrv), scores / scores.max(), color=colores_bar, alpha=0.8)
    ax.set(xlabel='Característica', ylabel='Score normalizado',
           title=f'{nombre}\nRojo = relevante real')
    ax.tick_params(axis='x', labelsize=7)

plt.suptitle('Selección de características — HRV para clasificación estrés/reposo\n'
              'Las características relevantes reales se muestran en rojo',
              fontsize=12, y=1.01)
plt.tight_layout()
plt.show()

## Parte 5 — Fuga en selección de características: demostración definitiva

Esta parte retoma el concepto de la Sesión 04 y lo aplica directamente
al pipeline de reducción de dimensionalidad — uno de los errores más
frecuentes en la literatura de neuroimagen y BCI.

In [ ]:
# ── Fuga en PCA: normalizar antes vs dentro de la CV ──────────────────────────
# Demostración: mismo error pero ahora con PCA como preprocesamiento

N_s, N_f = 120, 50
X_demo   = rng.normal(0, 1, (N_s, N_f))
y_demo   = rng.integers(0, 2, N_s)
# Sin señal real — cualquier exactitud > 50% es ficticia

K_CV    = 5
K_PCA   = 10   # componentes a retener
n_rep   = 80

accs_correcto, accs_fuga_pca = [], []

for _ in range(n_rep):
    Xr = rng.normal(0, 1, (N_s, N_f))
    yr = rng.integers(0, 2, N_s)
    idx_perm = rng.permutation(N_s)
    sz = N_s // K_CV

    acc_c_rep, acc_f_rep = [], []

    # Fuga: PCA ajustado en TODO el dataset
    Xr_c  = Xr - Xr.mean(axis=0)
    _, W_fuga, _, _ = pca_manual(Xr_c)
    Xr_pca_fuga = Xr_c @ W_fuga[:, :K_PCA]

    for k in range(K_CV):
        idx_te = idx_perm[k*sz:(k+1)*sz]
        idx_tr = np.concatenate([idx_perm[:k*sz], idx_perm[(k+1)*sz:]])

        # Correcto: PCA ajustado solo en train
        Xtr_c  = Xr[idx_tr] - Xr[idx_tr].mean(axis=0)
        Xte_c  = Xr[idx_te] - Xr[idx_tr].mean(axis=0)
        _, W_c, _, _ = pca_manual(Xtr_c)
        Xtr_p  = Xtr_c @ W_c[:, :K_PCA]
        Xte_p  = Xte_c @ W_c[:, :K_PCA]
        # Clasificador: umbral en el primer componente
        thr_c  = Xtr_p[:, 0].mean()
        pred_c = (Xte_p[:, 0] > thr_c).astype(int)
        acc_c_rep.append((pred_c == yr[idx_te]).mean())

        # Fuga: PCA ya ajustado en todo el dataset
        thr_f  = Xr_pca_fuga[idx_tr, 0].mean()
        pred_f = (Xr_pca_fuga[idx_te, 0] > thr_f).astype(int)
        acc_f_rep.append((pred_f == yr[idx_te]).mean())

    accs_correcto.append(np.mean(acc_c_rep))
    accs_fuga_pca.append(np.mean(acc_f_rep))

accs_correcto = np.array(accs_correcto)
accs_fuga_pca = np.array(accs_fuga_pca)

print('Fuga en PCA — datos sin señal real (cualquier exactitud > 50% es ficticia):')
print(f'  PCA correcto (dentro del fold): {accs_correcto.mean():.3f} ± {accs_correcto.std():.3f}')
print(f'  PCA con fuga (global antes CV): {accs_fuga_pca.mean():.3f} ± {accs_fuga_pca.std():.3f}')
print(f'  Inflación artificial: +{(accs_fuga_pca.mean()-accs_correcto.mean())*100:.1f} pp')

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(accs_correcto, bins=20, alpha=0.6, color='seagreen', density=True,
         label=f'PCA correcto μ={accs_correcto.mean():.2f}')
ax.hist(accs_fuga_pca, bins=20, alpha=0.6, color='tomato', density=True,
         label=f'PCA con fuga μ={accs_fuga_pca.mean():.2f}')
ax.axvline(0.5, color='k', ls='--', lw=1.5, label='Azar')
ax.set(xlabel='Exactitud CV', ylabel='Densidad',
       title='Fuga en PCA — datos puramente de ruido\n'
              'El PCA global crea ilusión de predectibilidad donde no existe')
ax.legend()
plt.tight_layout()
plt.show()

## ✏️ Ejercicios

1. **PCA y correlación.** Para el dataset EEG de 64 canales de la Parte 1, grafica
   el mapa de calor de la matriz de covarianza **antes** y **después** de aplicar PCA
   y retener los primeros 10 componentes. ¿Por qué la covarianza de los componentes
   principales es diagonal? ¿Qué propiedad de los vectores propios garantiza esto?

2. **LDA con regularización.** El LDA clásico falla cuando $N < d$ (más características
   que muestras — frecuente en EEG). Implementa LDA regularizado (RLDA) añadiendo
   $\gamma \mathbf{I}$ a $\mathbf{S}_W$ y realiza un barrido de $\gamma \in [10^{-4}, 10^2]$.
   Grafica la exactitud de validación cruzada vs $\gamma$ para el dataset de la Parte 2.

3. **ICA y eliminación de artefactos.** Extiende el ejemplo de la Parte 3 para:
   (a) eliminar el componente EMG (muscular — alta frecuencia) además del EOG,
   (b) comparar la señal alfa reconstruida antes y después de la eliminación
   mediante la correlación con la fuente original s1_alfa.

4. **Selección de características con CV correcta.** Para el dataset HRV de la
   Parte 4, implementa el pipeline correcto: selección de características
   (información mutua, top-10) **dentro** de cada fold del LOSO. Compara el AUROC
   resultante con la versión con fuga (selección global). ¿Cuánto se infla el AUROC?

5. *(Desafío)* **PCA en señales EEG reales.** Descarga el dataset PhysioNet EEG
   Motor Movement (physionet.org/content/eegmmidb/). Aplica el pipeline completo:
   segmentación → ICA para eliminación de artefactos → PCA para reducción
   de dimensionalidad → clasificación de imaginería motora con LOSO.
   Reporta el AUROC con IC95% bootstrap y compara con la línea base aleatoria
   usando la prueba de permutaciones.

## 📚 Conjuntos de datos utilizados / referenciados

| Conjunto de datos | Fuente | Notas |
|---|---|
| EEG Motor Movement | https://physionet.org/content/eegmmidb/ | 109 sujetos, 64 canales, imaginería motora |
| WESAD (HRV estrés) | Schmidt, P. et al. (2018). *ACM ICMI 2018*. https://doi.org/10.1145/3242969.3242985 | Diseño de características HRV |
| DREAMER (EEG/ECG) | https://zenodo.org/record/546113 | Reconocimiento de emociones — EEG multicanal |
| BCI Competition IV 2a | https://www.bbci.de/competition/iv/ | Imaginería motora 4 clases, 22 canales EEG |